# Aula 08 - Padrão de Design Multi-Agente


## Configuração


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Por que Sistemas Multi-Agentes?

Tarefas do mundo real, como planejamento de viagens, envolvem muitos tipos diferentes de especialização — logística, conhecimento local, orçamento e mais. Um único agente tentando lidar com tudo rapidamente se torna difícil de manejar.

Sistemas multi-agentes resolvem isso através da **especialização**: cada agente foca em uma área de especialidade, produzindo resultados de maior qualidade do que um generalista. Eles também melhoram a **escalabilidade** — você pode adicionar novos agentes (por exemplo, um especialista em voos, um crítico de restaurantes) sem reescrever o fluxo de trabalho existente. Os agentes se conectam por meio de um pipeline estruturado, passando o contexto de um para o outro.


## Criando Agentes Especializados


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Construindo um Fluxo de Trabalho Sequencial

`WorkflowBuilder` permite conectar agentes em um grafo direcionado. Aqui criamos um pipeline simples de duas etapas: o **TravelPlanner** elabora o itinerário, e então o **TravelConcierge** revisa e aprimora.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Adicionando Mais Agentes ao Fluxo de Trabalho

Uma das maiores vantagens do padrão multiagente é a facilidade de extensão. Abaixo adicionamos um agente **RevisorDeOrçamento** que verifica o plano em relação ao orçamento do viajante, sinaliza itens que podem exceder o limite de custos e sugere alternativas para economizar dinheiro. O fluxo de trabalho agora executa três agentes em sequência:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Resumo

Nesta lição, você aprendeu a:

1. **Criar agentes especializados** — cada um com um papel focado (planejamento, concierge, revisão orçamentária).
2. **Conectar agentes em um fluxo sequencial** usando `WorkflowBuilder` e `add_edge`.
3. **Transmitir a saída** de um pipeline multiagente, acompanhando qual agente está falando.
4. **Expandir um fluxo de trabalho** adicionando novos agentes à cadeia sem modificar os existentes.

O padrão de design multiagente mantém cada agente simples enquanto produz resultados mais ricos e cuidadosamente revisados do que qualquer agente individual poderia alcançar sozinho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
